# NLP: Lab 3 (Lemmatization/WordNet)

In [2]:
import nltk

nltk.download('twitter_samples')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')

from nltk.corpus import twitter_samples, stopwords, wordnet as wn
from nltk.tag import pos_tag
from nltk.stem.wordnet import WordNetLemmatizer
from nltk import FreqDist
import re, string

positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')

tweet_tokens = twitter_samples.tokenized('positive_tweets.json')
print(tweet_tokens[50])

[nltk_data] Downloading package twitter_samples to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['@groovinshawn', 'they', 'are', 'rechargeable', 'and', 'it', 'normally', 'comes', 'with', 'a', 'charger', 'when', 'u', 'buy', 'it', ':)']


# Task 0,  Execute the notebook and complete listed exercises (between CODE_START and CODE_END blocks).

In [3]:
def lemmatize_sentence(tokens):
    lemmatized_sentence = []
    lemmatizer = WordNetLemmatizer()

    # CODE_START
    for token, tag in pos_tag(tokens):
        if tag.startswith('NN'):
            pos = 'n'
        elif tag.startswith('VB'):
            pos = 'v'
        else:
            pos = 'a'
        lemmatized_sentence.append(lemmatizer.lemmatize(token, pos))
    # CODE_END

    return lemmatized_sentence

lemmatize_sentence(tweet_tokens[50])

['@groovinshawn',
 'they',
 'be',
 'rechargeable',
 'and',
 'it',
 'normally',
 'come',
 'with',
 'a',
 'charger',
 'when',
 'u',
 'buy',
 'it',
 ':)']

# Task 1 Change the code so it removes hashtags during pre-processing. (E.g. #Ukraine).

In [4]:
def process_tokens(tweet_tokens):
    cleaned_tokens = []
    stop_words = stopwords.words('english')
    lemmatizer = WordNetLemmatizer()

    for token, tag in pos_tag(tweet_tokens):
        # CODE_START
        # Task 1: також прибираємо хештеги (#word)
        if re.search(r'http[s]?://\S+|@\w+|#\w+', token):
            continue

        if token.lower() in stop_words or token in string.punctuation:
            continue

        if tag.startswith('NN'):
            pos = 'n'
        elif tag.startswith('VB'):
            pos = 'v'
        else:
            pos = 'a'

        cleaned_tokens.append(lemmatizer.lemmatize(token.lower(), pos))
        # CODE_END

    return cleaned_tokens

print("Before:", tweet_tokens[50])
print("After:", process_tokens(tweet_tokens[50]))

Before: ['@groovinshawn', 'they', 'are', 'rechargeable', 'and', 'it', 'normally', 'comes', 'with', 'a', 'charger', 'when', 'u', 'buy', 'it', ':)']
After: ['rechargeable', 'normally', 'come', 'charger', 'u', 'buy', ':)']


In [5]:
# CODE_START
positive_tweet_tokens = twitter_samples.tokenized('positive_tweets.json')
negative_tweet_tokens = twitter_samples.tokenized('negative_tweets.json')

positive_cleaned_tokens_list = [process_tokens(t) for t in positive_tweet_tokens]
negative_cleaned_tokens_list = [process_tokens(t) for t in negative_tweet_tokens]
# CODE_END

print(positive_tweet_tokens[500])
print(positive_cleaned_tokens_list[500])

['Dang', 'that', 'is', 'some', 'rad', '@AbzuGame', '#fanart', '!', ':D', 'https://t.co/bI8k8tb9ht']
['dang', 'rad', ':d']


In [6]:
def get_all_words(cleaned_tokens_list):
    # CODE_START
    for tokens in cleaned_tokens_list:
        yield from tokens
    # CODE_END

all_pos_words = get_all_words(positive_cleaned_tokens_list)

# CODE_START
freq_dist_pos = FreqDist(all_pos_words)
print(freq_dist_pos.most_common(10))
# CODE_END

[(':)', 3691), (':-)', 701), (':d', 658), ('thanks', 383), ('follow', 362), ('love', 336), ('...', 290), ('good', 283), ('get', 269), ('thank', 258)]


# Task 2 Modify process_tokens() so that instead of using lemmatizer.lemmatize(), it will use WordNet synsets.

In [7]:
def process_tokens_synset(tweet_tokens):
    cleaned_tokens = []
    stop_words = stopwords.words('english')

    for token, tag in pos_tag(tweet_tokens):
        if re.search(r'http[s]?://\S+|@\w+|#\w+', token):
            continue
        if token.lower() in stop_words or token in string.punctuation:
            continue

        # Замість lemmatize — беремо базову форму через synset
        synsets = wn.synsets(token.lower())
        if synsets:
            # Беремо lemma_names() першого (найпоширенішого) synset
            base_form = synsets[0].lemma_names()[0]
        else:
            base_form = token.lower()

        cleaned_tokens.append(base_form)

    return cleaned_tokens

print(process_tokens_synset(tweet_tokens[50]))

['rechargeable', 'normally', 'semen', 'charger', 'uracil', 'bargain', ':)']


# Taks 3  Let’s suppose that semantic distance between words is the distance to the common semantic parent (hypernym). Write a function that will compute this distance between two words.

In [8]:
def semantic_distance(word1, word2):
    """
    Відстань = кількість кроків до спільного гіпероніма.
    Використовуємо shortest_path_distance з WordNet.
    """
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)

    if not synsets1 or not synsets2:
        print(f"Одне зі слів не знайдено у WordNet.")
        return None

    # Беремо перший (найпоширеніший) synset для кожного слова
    s1 = synsets1[0]
    s2 = synsets2[0]

    # WordNet вбудований метод — відстань по ієрархії
    distance = s1.shortest_path_distance(s2)
    return distance

# Приклади:
print("car — automobile:", semantic_distance("car", "automobile"))  
print("human — monkey:",      semantic_distance("human", "monkey"))
print("car — tree:",       semantic_distance("car", "tree"))         
print("dog — cat:",        semantic_distance("dog", "cat"))

car — automobile: 0
human — monkey: 3
car — tree: 13
dog — cat: 4
